# 05 — Prepare three horizon-specific forecasting datasets

This notebook converts every selected direct target from all five tables into leakage-safe 1Q, 2Q, and 4Q examples. It preserves source meaning, geographic scope, compact origin-safe hierarchical context, and deterministic split boundaries. It prepares data only; model training remains in Notebook 06.


### What the setup code does and why

This code imports the required tools, locates the project folders, loads the shared evaluation configuration, and sets the random seed. We need this so panel construction uses the same horizons, history-window length, split dates, and reproducible settings as the rest of the project.

In [ ]:
# mount Google Drive and set the working directory to the project path
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/JobAI")  # the project path
os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"

In [ ]:
import hashlib, json, os
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / "configs" / "eval.yaml").is_file():
            return candidate
    return p

REPO = _find_repo()
PRO = REPO / "data" / "processed"
MAN = REPO / "data" / "manifests"
REPORTS = REPO / "reports"
for directory in (PRO, MAN, REPORTS):
    directory.mkdir(parents=True, exist_ok=True)
CFG = yaml.safe_load((REPO / "configs" / "eval.yaml").read_text())
HORIZONS = [int(h) for h in CFG["horizons"]]
WINDOW = int(CFG["feature_window_quarters"])
SEED = int(CFG["seed"])
np.random.seed(SEED)
print("repo:", REPO)
print("horizons:", HORIZONS, "window:", WINDOW)

## Preconditions and selected direct targets

**What this code does:** verifies normalization and baseline evidence, then loads every selected row explicitly marked as a direct forecast target. Benchmarks and geographic targets are no longer excluded from model training.


In [ ]:
selection_path = PRO / "selected_series.csv"
assertions_path = PRO / "normalization_assertions.json"
baseline_manifest_path = REPORTS / "baseline_run_manifest.json"
assert selection_path.is_file(), "Run notebook 03 first"
assert assertions_path.is_file(), "Run notebook 02 first"
assert baseline_manifest_path.is_file(), "Run updated notebook 04 first"
selection = pd.read_csv(selection_path)

def as_bool(series):
    return series.astype(str).str.lower().isin({"true", "1", "yes"})

selected_mask = as_bool(selection["selected"])
if "is_forecast_target" in selection:
    target_mask = as_bool(selection["is_forecast_target"])
else:
    target_mask = selection["role"].isin(["benchmark_target", "panel_target", "geographic_target"])
target_selection = selection[selected_mask & target_mask].copy()
assert not target_selection.empty, "No direct forecast targets passed notebook 03 selection"
normalization_assertions = json.loads(assertions_path.read_text())
assert all(result["status"] == "passed" for result in normalization_assertions.values())
baseline_manifest = json.loads(baseline_manifest_path.read_text())
selection_sha = hashlib.sha256(selection_path.read_bytes()).hexdigest()
assert baseline_manifest.get("selection_catalog_sha256") == selection_sha, "Rerun notebook 04 after notebook 03"
display(target_selection.groupby(["table_id", "forecast_scope"]).size().rename("n_series").reset_index())
print("selected direct forecast series:", len(target_selection))


## Construct leakage-safe examples and compact hierarchy context

For every target and forecast origin, this code keeps only historical values available at that origin. It also adds a compact reference trend: official national ATP context for broad ATP regions, same-occupation or same-industry national aggregates for detailed KEHA panels, and the matching `12r5` whole-country series for detailed geographies.


In [ ]:
splits = CFG["splits"]
FEATURE_SET = CFG.get("engineered_feature_set", "enhanced_v1")

def quarter_ordinal(q):
    return int(q[:4]) * 4 + int(q[-1]) - 1

def assigned_split(origin, target):
    if origin <= splits["train_end"] and target <= splits["train_end"]:
        return "train"
    if splits["val_start"] <= origin <= splits["val_end"] and target <= splits["val_end"]:
        return "validation"
    if splits["test_start"] <= origin <= splits["test_end"]:
        return "test"
    return None

def slope(values):
    values = np.asarray(values, dtype=float)
    x = np.arange(len(values), dtype=float)
    return float(np.polyfit(x, values, 1)[0]) if len(values) > 1 else 0.0

def engineered_features(window_values, origin_quarter, scale):
    values = np.asarray(window_values, dtype=float)
    recent = values[-4:]
    return {
        "quarter_of_year": int(origin_quarter[-1]),
        "qoq_change_scaled": float((values[-1] - values[-2]) / scale),
        "yoy_change_scaled": float((values[-1] - values[-5]) / scale),
        "recent_mean_4_scaled": float((np.mean(recent) - values[-1]) / scale),
        "window_mean_scaled": float((np.mean(values) - values[-1]) / scale),
        "recent_slope_4_scaled": float(slope(recent) / scale),
        "window_slope_scaled": float(slope(values) / scale),
        "recent_std_4_scaled": float(np.std(recent) / scale),
        "window_std_scaled": float(np.std(values) / scale),
        "zero_fraction_window": float(np.mean(values == 0)),
    }

def quarter_value_map(frame):
    values = pd.to_numeric(frame["value"], errors="coerce")
    return dict(zip(frame["timeperiod_q"].astype(str), values))

def size_band(value):
    value = abs(float(value))
    if value == 0: return "zero"
    if value <= 25: return "1-25"
    if value <= 100: return "26-100"
    if value <= 500: return "101-500"
    return "over-500"

def volatility_band(ratio):
    if ratio <= 0.10: return "low"
    if ratio <= 0.30: return "medium"
    if ratio <= 1.00: return "high"
    return "very_high"

tables = {table_id: pd.read_csv(PRO / f"{table_id}__normalized.csv", low_memory=False)
          for table_id in sorted(target_selection["table_id"].unique())}

# Build origin-safe reference series. No future target values are copied into a prompt.
context_maps = {}
for content, group in tables["11l1"].groupby("contentscode", sort=False):
    context_maps[("11l1", str(content))] = quarter_value_map(group)

tu_context = tables["12tu"].copy()
tu_context["value"] = pd.to_numeric(tu_context["value"], errors="coerce")
tu_context = tu_context[(tu_context["Työmarkkina-asema"].astype(str) == "SSS")]
tu_group_cols = ["Ammattiryhmä", "Työmarkkina-asema", "contentscode"]
tu_aggregate = tu_context.groupby(tu_group_cols + ["timeperiod_q"], dropna=False)["value"].sum(min_count=1).reset_index()
for keys, group in tu_aggregate.groupby(tu_group_cols, dropna=False, sort=False):
    context_maps[("12tu", *map(str, keys))] = quarter_value_map(group)

tw_context = tables["12tw"].copy()
tw_context["value"] = pd.to_numeric(tw_context["value"], errors="coerce")
tw_group_cols = ["Toimiala", "Työnantajan sektori", "Työpaikan työn kesto", "contentscode"]
tw_aggregate = tw_context.groupby(tw_group_cols + ["timeperiod_q"], dropna=False)["value"].sum(min_count=1).reset_index()
for keys, group in tw_aggregate.groupby(tw_group_cols, dropna=False, sort=False):
    context_maps[("12tw", *map(str, keys))] = quarter_value_map(group)

r5_national = tables["12r5"][tables["12r5"]["Alue"].astype(str) == "SSS"]
for content, group in r5_national.groupby("contentscode", sort=False):
    context_maps[("12r5", str(content))] = quarter_value_map(group)

def context_reference(table_id, dimensions):
    if table_id in ("11l1", "11n1"):
        return ("11l1", str(dimensions["contentscode"])), "official_national_atp"
    if table_id == "12tu":
        key = ("12tu", str(dimensions["Ammattiryhmä"]), str(dimensions["Työmarkkina-asema"]), str(dimensions["contentscode"]))
        return key, "national_same_occupation_keha"
    if table_id == "12tw":
        key = ("12tw", str(dimensions["Toimiala"]), str(dimensions["Työnantajan sektori"]),
               str(dimensions["Työpaikan työn kesto"]), str(dimensions["contentscode"]))
        return key, "national_same_industry_keha"
    return ("12r5", str(dimensions["contentscode"])), "national_same_measure_keha"

examples = []
skipped_missing_window = 0
skipped_missing_target = 0
for selected in target_selection.itertuples(index=False):
    dimensions = json.loads(selected.dimensions_json)
    frame = tables[selected.table_id]
    for column, value in dimensions.items():
        frame = frame[frame[column].astype(str) == str(value)]
    frame = frame[["timeperiod_q", "value"]].copy()
    frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
    frame["quarter_index"] = frame["timeperiod_q"].map(quarter_ordinal)
    frame = frame.sort_values("quarter_index").reset_index(drop=True)
    assert frame["timeperiod_q"].is_unique
    assert np.diff(frame["quarter_index"]).tolist() == [1] * (len(frame) - 1)
    quarters = frame["timeperiod_q"].tolist()
    values = frame["value"].to_numpy(dtype=float)
    reference_key, reference_name = context_reference(selected.table_id, dimensions)
    reference_map = context_maps.get(reference_key, {})
    for origin_idx in range(WINDOW - 1, len(values)):
        window_quarters = quarters[origin_idx - WINDOW + 1:origin_idx + 1]
        window_values = values[origin_idx - WINDOW + 1:origin_idx + 1]
        if not np.isfinite(window_values).all():
            skipped_missing_window += len(HORIZONS)
            continue
        origin = quarters[origin_idx]
        last_value = float(window_values[-1])
        window_mean = float(np.mean(window_values))
        window_std = float(np.std(window_values))
        scale = max(abs(last_value), window_std, 1.0)
        normalized_window = ((window_values - last_value) / scale).tolist()
        feature_values = engineered_features(window_values, origin, scale)
        reference_values = np.asarray([reference_map.get(q, np.nan) for q in window_quarters], dtype=float)
        context_payload = {}
        if np.isfinite(reference_values).all():
            reference_scale = max(abs(float(reference_values[-1])), float(np.std(reference_values)), 1.0)
            context_payload = {
                "reference": reference_name,
                "latest": float(reference_values[-1]),
                "qoq_scaled": float((reference_values[-1] - reference_values[-2]) / reference_scale),
                "yoy_scaled": float((reference_values[-1] - reference_values[-5]) / reference_scale),
                "slope4_scaled": float(slope(reference_values[-4:]) / reference_scale),
            }
        for horizon in HORIZONS:
            target_idx = origin_idx + horizon
            if target_idx >= len(values) or not np.isfinite(values[target_idx]):
                skipped_missing_target += 1
                continue
            target_quarter = quarters[target_idx]
            split = assigned_split(origin, target_quarter)
            if split is None:
                continue
            target_value = float(values[target_idx])
            identity = f"{selected.series_id}|{origin}|h{horizon}"
            volatility_ratio = window_std / max(abs(window_mean), 1.0)
            examples.append({
                "example_id": hashlib.sha256(identity.encode()).hexdigest()[:20],
                "split": split, "table_id": selected.table_id, "series_family": selected.series_family,
                "forecast_scope": selected.forecast_scope, "measure_code": selected.measure_code,
                "series_id": selected.series_id, "dimensions_json": selected.dimensions_json,
                "origin_quarter": origin, "target_quarter": target_quarter, "horizon_q": horizon,
                "window_start_quarter": window_quarters[0],
                "input_values_json": json.dumps([float(x) for x in window_values]),
                "normalized_input_json": json.dumps([float(x) for x in normalized_window]),
                "feature_set": FEATURE_SET,
                "engineered_features_json": json.dumps(feature_values, sort_keys=True),
                "hierarchical_context_json": json.dumps(context_payload, ensure_ascii=False, sort_keys=True),
                "last_value": last_value, "window_mean": window_mean, "window_std": window_std, "scale": scale,
                "target_value": target_value,
                "target_scaled_change": (target_value - last_value) / scale,
                "target_log_change": float(np.log1p(target_value) - np.log1p(last_value)),
                "target_size_band": size_band(target_value),
                "volatility_band": volatility_band(volatility_ratio),
            })
dataset = pd.DataFrame(examples).sort_values(["split", "horizon_q", "origin_quarter", "table_id", "series_id"]).reset_index(drop=True)
assert dataset["example_id"].is_unique
assert dataset["engineered_features_json"].notna().all()
assert (dataset["target_quarter"].map(quarter_ordinal) > dataset["origin_quarter"].map(quarter_ordinal)).all()
assert (dataset.loc[dataset.split == "train", "target_quarter"] <= splits["train_end"]).all()
assert (dataset.loc[dataset.split == "validation", "target_quarter"] <= splits["val_end"]).all()
print("feature set:", FEATURE_SET)
print("examples:", len(dataset))
print("skipped missing-window attempts:", skipped_missing_window)
print("skipped missing targets:", skipped_missing_target)
display(dataset.groupby(["split", "horizon_q", "table_id", "forecast_scope"]).size().rename("n_examples").reset_index())


## Persist combined and horizon-specific splits

The combined files remain available for audits, while the new `h1`, `h2`, and `h4` files provide an explicit input for each adapter. The dataset card records every checksum and the count available for each horizon.


### Record provenance and decide whether fine-tuning is allowed

This code records file paths, row counts, checksums, split boundaries, and dataset settings in a dataset card. It then checks the required number of selected series and training examples. Fine-tuning is marked ready only when normalization passed, the baseline evidence exists, and both data-size requirements pass.

In [ ]:
combined_path = PRO / "panel_forecasting_dataset.parquet"
dataset.to_parquet(combined_path, index=False)
split_paths = {}
horizon_paths = {}
for split in ("train", "validation", "test"):
    split_frame = dataset[dataset["split"] == split].copy()
    path = PRO / f"panel_{split}.jsonl"
    split_frame.to_json(path, orient="records", lines=True, force_ascii=False)
    split_paths[split] = path
    for horizon in HORIZONS:
        horizon_frame = split_frame[split_frame["horizon_q"] == horizon].copy()
        horizon_path = PRO / f"panel_{split}_h{horizon}.jsonl"
        horizon_frame.to_json(horizon_path, orient="records", lines=True, force_ascii=False)
        horizon_paths[(split, horizon)] = horizon_path

summary = (dataset.groupby(["split", "table_id", "series_family", "forecast_scope", "horizon_q"])
           .agg(n_examples=("example_id", "size"), n_series=("series_id", "nunique"),
                first_origin=("origin_quarter", "min"), last_origin=("origin_quarter", "max"),
                first_target=("target_quarter", "min"), last_target=("target_quarter", "max"))
           .reset_index())
summary_path = REPORTS / "panel_dataset_summary.csv"
summary.to_csv(summary_path, index=False)
display(summary)

gate_cfg = CFG["finetuning_gate"]
gate_checks = {
    "normalization_assertions_passed": all(result["status"] == "passed" for result in normalization_assertions.values()),
    "baseline_report_present": baseline_manifest_path.is_file(),
    "selected_panel_series": int(target_selection["series_id"].nunique()),
    "minimum_selected_panel_series": int(gate_cfg["minimum_selected_panel_series"]),
    "train_examples": int((dataset["split"] == "train").sum()),
    "minimum_train_examples": int(gate_cfg["minimum_train_examples"]),
}
gate_checks["series_gate_passed"] = gate_checks["selected_panel_series"] >= gate_checks["minimum_selected_panel_series"]
gate_checks["example_gate_passed"] = gate_checks["train_examples"] >= gate_checks["minimum_train_examples"]
gate_checks["finetuning_ready"] = bool(gate_checks["normalization_assertions_passed"] and gate_checks["baseline_report_present"] and gate_checks["series_gate_passed"] and gate_checks["example_gate_passed"])
print("fine-tuning gate:", "PASS" if gate_checks["finetuning_ready"] else "FAIL")
print(json.dumps(gate_checks, indent=2))


In [ ]:
def file_record(path, rows):
    return {"path": str(path.relative_to(REPO)), "rows": int(rows), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}

horizon_counts = {
    split: {str(horizon): int(((dataset["split"] == split) & (dataset["horizon_q"] == horizon)).sum()) for horizon in HORIZONS}
    for split in ("train", "validation", "test")
}
manifest = {
    "name": "JobAI five-table direct-target forecasting dataset",
    "responsibility": "dataset construction only; no model training",
    "feature_window_quarters": WINDOW, "horizons_q": HORIZONS, "splits": CFG["splits"],
    "feature_set": FEATURE_SET,
    "engineered_features": list(CFG.get("engineered_features", [])),
    "hierarchical_context": "origin_safe_compact_v1",
    "selected_panel_series": int(target_selection["series_id"].nunique()),
    "selected_series_by_table": target_selection.groupby("table_id").size().astype(int).to_dict(),
    "selection_catalog_sha256": hashlib.sha256(selection_path.read_bytes()).hexdigest(),
    "baseline_manifest_sha256": hashlib.sha256(baseline_manifest_path.read_bytes()).hexdigest(),
    "horizon_counts": horizon_counts,
    "gate": gate_checks,
    "files": {"combined": file_record(combined_path, len(dataset))},
    "horizon_files": {},
}
for split, path in split_paths.items():
    manifest["files"][split] = file_record(path, (dataset["split"] == split).sum())
for horizon in HORIZONS:
    manifest["horizon_files"][str(horizon)] = {}
    for split in ("train", "validation", "test"):
        path = horizon_paths[(split, horizon)]
        manifest["horizon_files"][str(horizon)][split] = file_record(path, horizon_counts[split][str(horizon)])
manifest_path = MAN / "panel_dataset_card.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
for path in [combined_path, *split_paths.values(), *horizon_paths.values(), summary_path, manifest_path]:
    print("wrote:", path)


## Fine-tuning decision

When the gate passes, Notebook 06 should be run three times with `model_qwen3_4b_h1.yaml`, `model_qwen3_4b_h2.yaml`, and `model_qwen3_4b_h4.yaml`. Each run uses one horizon only and writes a new immutable adapter directory. The untouched test files are reserved for Notebook 07.
